# Zones

Football pitches are often divided into zones. What those zones look like may vary from person to person. The FMA library implements the 20 zones made popular by Pep Guardiola. This tutorial will teach out how you can explore those zones in the FMA library.

---

**Change Directory**

This block of code is only ran so that, so this notebook can use the FMA library.

In [ ]:
from pathlib import Path
import sys
import os

cwd = os.getcwd()
os.chdir(cwd)

project_root = Path.cwd().parent
sys.path.append(str(project_root))

**Import FMA Library**

In [ ]:
from FootballMatchAnalysis.objects.match import Match
from FootballMatchAnalysis.objects.plot import Plot

**Load Event and Tracking Data**

In [ ]:
DATADIR = './../data'
game_id = 1

In [ ]:
match = Match(DATADIR, game_id)

**Import the Zones Library**

The FMA library includes a series of libraries that you can utilize to support your analysis. We will start by importing the `zones` library.

In [ ]:
from FootballMatchAnalysis.analysis.zones import *

**Zones on a Pitch**

To draw all the zones on a pitch, we will use `get_zones` to get the raw bounding boxes for each zone and draw them onto a pitch.

In [ ]:
# Plot
plot = Plot()

# Get Zones
zones = get_zones()

# Iterate Over Each Zone
for zone_number in zones:
    # Get and Draw Zone
    zone = zones[zone_number]
    plot.draw_box(zone)

    # Label Zones
    x_center = ((zone[0][0] - zone[2][0]) / 2) + zone[2][0]
    y_center = ((zone[0][1] - zone[1][1]) / 2) + zone[1][1]
    plot.write(zone_number, x_center, y_center)

**Finding Which Zones Events Took**

In [ ]:
# Get a Pass
passes = match.get_events("PASS")
p = passes.iloc[100]

## What Zone Did The Pass Start In?
# Get Starting and Ending Coordinates
start = (p["Start X"], p["Start Y"])
end = (p["End X"], p["End Y"])

#What Zone Did the Pass Start In?
zone = what_zone(start)
print(f"The pass started in zone {zone}")

#What Zone Did the Pass End In?
zone = what_zone(end)
print(f"The pass ended in zone {zone}")

In [ ]:
plot.draw_event(p)

**Profiling Players Using Zones**

Analyzing which zones specific actions took place can be an important tool in profiling players. For example, Team 10 will likely be playing passes into Zone 20. Let's find which player has done exactly that.

In [ ]:
# Get All Passes
passes = match.get_events("PASS")

# Iterate Over Each Pass
players = {}
for i, p in passes.iterrows():
    # Get Start and End Coordinates
    start = (p["Start X"], p["Start Y"])
    end = (p["End X"], p["End Y"])

    # Find Passes into Zone 20
    if what_zone(start) != "20" and what_zone(end) == "20":
        player = p["From"]
        
        if player not in players:
            players[player] = []
        players[player].append(p)

# Find Player Who Played Most Passes into Zone 20
player = max(players, key=lambda p: len(players[p]))
print(f"{player} played the most passes into Zone 20")


# Visualize Passes
plot = Plot()
for zone_number in get_zones():
    # Get and Draw Zone
    zone = zones[zone_number]
    plot.draw_box(zone)

for p in players[player]:
    plot.draw_event(p)